# Hyperparameter optimization: results

The PCA sweep fixes how many components the equilibrium target keeps; the search then tunes
the network that regresses it. Both cost hours, so this notebook reads their tables
(`pca_component_sweep.csv`, `pca_explained_variance.csv`, the newest search's `trials.csv`).
Method: `mhd_equilibrium_forward_models/hyperparameter_optimization/hyperparameter_optimization_utils.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    model_configs,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models.hyperparameter_optimization import (  # noqa: E501
    hyperparameter_optimization_utils,
    pca_component_sweep,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    pubfig,
)

# Anchored to the package, not the kernel's working directory.
OUTPUT_DIR = hyperparameter_optimization_utils.RESULTS_DIRECTORY
PLOTS_DIR = OUTPUT_DIR / "plots"
DATA_DIR = hyperparameter_optimization_utils.DATA_DIRECTORY
MODEL = hyperparameter_optimization_utils.MODEL

pubfig.apply_style()
print(OUTPUT_DIR)

## 1. How many PCA components should the target keep?

One surrogate per width, everything else fixed, each scored against the round trip of its own
representation (the exact target projected through the same PCA). Run
`pca_component_sweep.py --n-workers 4` first.

In [ ]:
sweep_path = OUTPUT_DIR / pca_component_sweep.SWEEP_FILE
assert sweep_path.exists(), f"run pca_component_sweep.py first; no {sweep_path.name}"
sweep = pd.read_csv(sweep_path)

# Each physical metric beside its representation floor. `validation_mse` is left out:
# every width measures it in its own latent space, so a narrower space looks better for
# that reason alone.
pca_columns = [
    "latent_width",
    "flux_surface_error_m",
    "flux_surface_error_floor_m",
    "fnorm",
    "fnorm_floor",
    "iota_rmse",
    "n_parameters",
]
pca_table = sweep.sort_values("latent_width").loc[:, pca_columns].reset_index(drop=True)
# The target's own residual is a property of the data, so it must not vary by row.
assert sweep["fnorm_target"].nunique() == 1, "target residual differs between rows"
print(f"DESC solves' own force residual: {sweep['fnorm_target'].iloc[0]:.3e}")
pca_table

In [ ]:
variance_path = OUTPUT_DIR / "pca_explained_variance.csv"
if variance_path.exists():
    variance_curve = pd.read_csv(variance_path)
else:
    variance_curve = pca_component_sweep.explained_variance_curve(DATA_DIR)
    variance_curve.to_csv(variance_path, index=False)

figure, axes = pubfig.subplots(width="single", aspect=0.75)
panel = axes[0]
components = np.asarray(variance_curve["n_components"])
explained = np.asarray(variance_curve["explained_variance"])
panel.plot(components, explained, color=pubfig.PALETTE[0], lw=1.2)
# Marked at the width in use. The project pins a component count, not a variance
# fraction, because asking sklearn for a fraction is not reproducible.
n_components = model_configs.N_PCA_COMPONENTS
reached = float(explained[np.searchsorted(components, n_components)])
panel.axvline(n_components, color=pubfig.PALETTE[1], ls="--", lw=0.9)
# Placed left of its own line and right-aligned, so a mark near the right edge
# keeps its label inside the axes.
panel.annotate(
    f"{n_components} components, {reached:.6f}",
    xy=(n_components, 0.0),
    xycoords=("data", "axes fraction"),
    xytext=(-4, 8),
    textcoords="offset points",
    ha="right",
    color=pubfig.PALETTE[1],
    fontsize="small",
)
panel.set_xscale("log")
panel.set_xlabel("PCA components")
panel.set_ylabel("cumulative explained variance")
pubfig.save(figure, str(PLOTS_DIR / "pca_explained_variance"))

print(f"{n_components} components explain {reached:.7f}")

## 2. Network hyperparameters

From the newest search's run folder. Lower `objective` is better; `objective_term` names the
validation term. A running best that flattens before the last trial means the search converged
within its budget. Heatmap numbers are trial counts: nevergrad concentrates trials where it
finds improvements, so a one-trial cell is a sample, not a measurement.

In [ ]:
runs = sorted(hyperparameter_optimization_utils.RUNS_DIRECTORY.glob(f"*_{MODEL}"))
assert runs, f"no search of {MODEL} yet; run hyperparameter_optimization_utils.py first"
search_directory = runs[-1]
trials = pd.read_csv(search_directory / hyperparameter_optimization_utils.TRIALS_FILE)
# A search ranks on its last stage's term. A curriculum ending on `fnorm` minimizes a
# logarithm, so its objective is negative and needs its own label.
objective_labels = {
    "total": "validation MSE",
    "data": "validation MSE",
    "fnorm": "validation log force residual",
}
objective_term = str(trials["objective_term"].iloc[0])
objective_label = objective_labels.get(objective_term, f"validation {objective_term}")
best_columns = [
    "trial",
    "width",
    "depth",
    "activation",
    "learning_rate",
    "weight_decay",
    "dropout_rate",
    "batch_size",
    "objective",
    "n_epochs",
    "n_parameters",
]
# The five best trials, best first (`.loc` includes its end label).
best = trials.sort_values("objective", ignore_index=True).loc[:4, best_columns]
print(f"{search_directory.name}: {len(trials)} trials")
best.head()

In [ ]:
figure, axes = pubfig.subplots(width="single", aspect=0.75)
panel = axes[0]
panel.scatter(
    trials["trial"],
    trials["objective"],
    s=18,
    color=pubfig.PALETTE[0],
    label="trial",
)
panel.step(
    trials["trial"],
    trials["objective"].cummin(),
    where="post",
    color="k",
    lw=1.2,
    label="running best",
)
panel.set_xlabel("trial")
panel.set_ylabel(objective_label)
panel.legend()
pubfig.save(figure, str(PLOTS_DIR / "hp_search_convergence"))

In [ ]:
n_width_bins = 5
widths = np.asarray(trials["width"], dtype=float)
depths = np.asarray(trials["depth"], dtype=int)
objective = np.asarray(trials["objective"], dtype=float)

rows = np.unique(depths)
edges = np.linspace(widths.min(), widths.max(), n_width_bins + 1)
column_of_trial = np.clip(np.digitize(widths, edges) - 1, 0, n_width_bins - 1)

mean_objective = np.full((rows.size, n_width_bins), np.nan)
counts = np.zeros((rows.size, n_width_bins), dtype=int)
for row, depth in enumerate(rows):
    for column in range(n_width_bins):
        selected = (depths == depth) & (column_of_trial == column)
        counts[row, column] = int(selected.sum())
        if selected.any():
            mean_objective[row, column] = float(objective[selected].mean())

figure, axes = pubfig.subplots(width="single", aspect=0.85)
panel = axes[0]
colormap = plt.get_cmap("viridis").copy()
colormap.set_bad("0.9")
mesh = panel.pcolormesh(
    edges,
    np.append(rows - 0.5, rows[-1] + 0.5),
    np.ma.masked_invalid(mean_objective),
    cmap=colormap,
)
figure.colorbar(mesh, ax=panel, label=f"mean {objective_label}")

finite = mean_objective[np.isfinite(mean_objective)]
midpoint = 0.5 * (finite.min() + finite.max()) if finite.size else 0.0
centres = 0.5 * (edges[:-1] + edges[1:])
for row, depth in enumerate(rows):
    for column in range(n_width_bins):
        if counts[row, column] == 0:
            continue
        panel.text(
            centres[column],
            depth,
            str(counts[row, column]),
            ha="center",
            va="center",
            fontsize="small",
            color="w" if mean_objective[row, column] < midpoint else "k",
        )

panel.set_yticks(rows)
panel.set_xlabel("hidden layer width")
panel.set_ylabel("hidden layers")
pubfig.save(figure, str(PLOTS_DIR / "hp_width_depth_heatmap"))